In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pricing-ab01/pricing.csv


Business Problem
Sell items to users in a game company game
gave gift money for their purchases.
Users can use these virtual coins to
buys various vehicles.
The game company has not specified a price for an item and
users to buy this item at the price they want
provided.
For example, for an item named shield, users can
They will buy this shield by paying the amounts they see.
In other words, a user has 30 virtual moneys
unit, other user can pay with 45 units.
Therefore, users can afford to pay according to them.
They can buy this item with the amounts they have bought. 
1- The price of the item varies by category
is it? Express it statistically.
2- Depending on the first question, what should the price of the item be?
Explain why?
3- "To be able to move" about the price
is requested. Decision support for pricing strategy
create the system.
4- Purchase items for possible price changes and
Simulate their income. 

In [2]:
df=pd.read_csv("../input/pricing-ab01/pricing.csv",sep=";")

In [3]:
df.head()

,category_id,price
0,489756,32.117753
1,361254,30.711370
2,361254,31.572607
3,489756,34.543840
4,489756,47.205824


In [4]:
df.isnull().sum().any()

False

In [5]:

    
def outlier_thresholds(dataframe, variable):
    quartile1 = dataframe[variable].quantile(0.05)
    quartile3 = dataframe[variable].quantile(0.95)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

In [7]:
replace_with_thresholds(df,"price")

In [8]:
from scipy.stats import shapiro

In [9]:
for i in df.category_id.unique():
    print(i)

489756
361254
874521
326584
675201
201436


In [10]:
shapiro(df.loc[df.category_id==489756,"price"])

ShapiroResult(statistic=0.5525132417678833, pvalue=0.0)

In [11]:
shapiro(df.loc[df.category_id==361254,"price"])

ShapiroResult(statistic=0.3058021664619446, pvalue=4.680336870844889e-43)

In [12]:
shapiro(df.loc[df.category_id==874521,"price"])

ShapiroResult(statistic=0.45945078134536743, pvalue=1.3438452272874996e-42)

In [13]:
shapiro(df.loc[df.category_id==675201,"price"])

ShapiroResult(statistic=0.4161914587020874, pvalue=1.2883891379106764e-20)

In [14]:
shapiro(df.loc[df.category_id==326584,"price"])

ShapiroResult(statistic=0.3980863094329834, pvalue=6.487847549253148e-22)

In [15]:
shapiro(df.loc[df.category_id==201436,"price"])

ShapiroResult(statistic=0.6189789772033691, pvalue=1.7088095193950985e-14)

If the normality of any of them is not provided, there is no need to apply variance homogeneity test to be applied Mannwhitney, normality is not provided. 

In [16]:
from itertools import combinations

In [17]:
komb=[]
for i in combinations(df["category_id"].unique(),2):
    komb.append(i)
komb

[(489756, 361254),
 (489756, 874521),
 (489756, 326584),
 (489756, 675201),
 (489756, 201436),
 (361254, 874521),
 (361254, 326584),
 (361254, 675201),
 (361254, 201436),
 (874521, 326584),
 (874521, 675201),
 (874521, 201436),
 (326584, 675201),
 (326584, 201436),
 (675201, 201436)]

In [18]:
from scipy.stats import stats 

In [19]:
komb2=[]
komb3=[]
for i in komb:
    teststatistic,pvalue=stats.mannwhitneyu(df.loc[df.category_id==i[0],"price"],
                                       df.loc[df.category_id==i[1],"price"])
    if pvalue<0.05:
        komb2.append([i[0],i[1],"h0 rejection"])
    else:
        komb3.append([i[0],i[1],"h0 irrefutable "])

In [20]:
komb2

[[489756, 361254, 'h0 rejection'],
 [489756, 874521, 'h0 rejection'],
 [489756, 326584, 'h0 rejection'],
 [489756, 675201, 'h0 rejection'],
 [489756, 201436, 'h0 rejection'],
 [361254, 874521, 'h0 rejection'],
 [361254, 326584, 'h0 rejection'],
 [874521, 326584, 'h0 rejection'],
 [326584, 675201, 'h0 rejection'],
 [326584, 201436, 'h0 rejection']]

In [21]:
komb3

[[361254, 675201, 'h0 irrefutable '],
 [361254, 201436, 'h0 irrefutable '],
 [874521, 675201, 'h0 irrefutable '],
 [874521, 201436, 'h0 irrefutable '],
 [675201, 201436, 'h0 irrefutable ']]

h0 there is no statistically significant difference between them
There is a significant statistical difference between h1 (mannwhitneyu)
There is no difference between the two categories, so we can make the same charges. 